In [1]:
from cs336_basics.module.transformer import TransformerConfig
from cs336_basics.resource_accounting import model_params, flops, print_resource_summary

In [2]:
GPT_2_XL = TransformerConfig(vocab_size=50257, context_length=1024, num_layers=48, d_model=1600, num_heads=25, d_ff=6400)
print(f"Total model params is {model_params(GPT_2_XL):_}, total flops is {flops(GPT_2_XL):_}")

TransformerConfig: TransformerConfig(vocab_size=50257, context_length=1024, d_model=1600, d_ff=6400, rope_theta=10000.0, num_layers=48, num_heads=25)
Attention has 491_520_000 params, Feed forward 1_474_560_000 params, out linear 80_411_200 params, emb 80_411_200 params
Attention consumes 25.00% flops, Feed forward 75.00% flops, and out linear 0.00% flops
Total model params is 2_127_057_600, total flops is 4_026_692_662_400


# Summary
- GPT-2 XL using our model arch would have 2.1B params, 4TFLOPS. Using float32, it would require 8.4GB memory to load the model.

This is slighly higher than the 1.6B params and 3.2 TFLOPS from the original GPT-2 XL model as SwiGLU has 3 matrix multiplies as opposed to 2 in the tranditional GPT-2 MLP layer. 

- The total of flops scale linearly with context_length and quadratically with d_model.

- MLP always consumes most flops, for about *75%* of total flops in this case with `d_ff = d_model * 4`, or *67%* with `d_ff = d_model * 8/3`

In [3]:
models = {
    "GPT_2_S": TransformerConfig(vocab_size=50257, context_length=1024, num_layers=12, d_model=768, num_heads=12, d_ff=4*768),
    "GPT_2_M": TransformerConfig(vocab_size=50257, context_length=1024, num_layers=24, d_model=1024, num_heads=16, d_ff=4*1024),
    "GPT_2_L": TransformerConfig(vocab_size=50257, context_length=1024, num_layers=36, d_model=1280, num_heads=20, d_ff=4*1280),
    "GPT_2_XL": TransformerConfig(vocab_size=50257, context_length=1024, num_layers=48, d_model=1600, num_heads=25, d_ff=4*1600),
    "GPT_2_XL_extra_context_length": TransformerConfig(vocab_size=50257, context_length=16384, num_layers=48, d_model=1600, num_heads=25, d_ff=4*1600),
}

for model, config in models.items():
    print(f"Analyzing flops for {model}")
    print(f"total flops is {flops(config):_}")
    print()
    

TransformerConfig: TransformerConfig(vocab_size=50257, context_length=1024, d_model=768, d_ff=3072, rope_theta=10000.0, num_layers=12, num_heads=12)
TransformerConfig: TransformerConfig(vocab_size=50257, context_length=1024, d_model=1024, d_ff=4096, rope_theta=10000.0, num_layers=24, num_heads=16)
TransformerConfig: TransformerConfig(vocab_size=50257, context_length=1024, d_model=1280, d_ff=5120, rope_theta=10000.0, num_layers=36, num_heads=20)
TransformerConfig: TransformerConfig(vocab_size=50257, context_length=1024, d_model=1600, d_ff=6400, rope_theta=10000.0, num_layers=48, num_heads=25)
TransformerConfig: TransformerConfig(vocab_size=50257, context_length=16384, d_model=1600, d_ff=6400, rope_theta=10000.0, num_layers=48, num_heads=25)
Analyzing flops for GPT_2_S
Attention consumes 24.99% flops, Feed forward 74.98% flops, and out linear 0.03% flops
total flops is 232_005_428_736

Analyzing flops for GPT_2_M
Attention consumes 25.00% flops, Feed forward 74.99% flops, and out linear 